# Machine Learning — Attitudes Towards Feminism

## Research question

Can sociodemographic characteristics help predict whether a respondent
agrees that feminism threatens the place and role of men in society?

## Objective

This classification model explores whether selected characteristics
are associated with responses to this statement in the 2025 HCE
sexism survey.

The results may help CIDFF reflect on its awareness activities.
The model is not intended to identify dangerous individuals or to
explain the causes of their opinions.

In [1]:
import pandas as pd

# Load the 2025 HCE survey
hce = pd.read_csv("base-de-donnees-toluna-harris-barometre-sexisme-vague-5-2025-dgcs-hce-.csv",sep=";",dtype="string")

print("Survey shape:", hce.shape)

Survey shape: (3061, 264)


## Step 1 — Exploratory Data Analysis

Explore the survey question and the characteristics selected for the model.

In [2]:
# Find the survey question used as our target
target_column = next(
    column for column in hce.columns
    if column.startswith("Q4_19 :"))

print(target_column)
print("\nResponses:")
print(hce[target_column].value_counts(dropna=False))

Q4_19 : Le féminisme menace la place et le rôle des hommes dans la société (Q4 - Dans quelle mesure êtes-vous d’accord avec les affirmations suivantes ?)

Responses:
Q4_19 : Le féminisme menace la place et le rôle des hommes dans la société (Q4 - Dans quelle mesure êtes-vous d’accord avec les affirmations suivantes ?)
Pas du tout d’accord         948
Plutôt pas d’accord          858
Plutôt d’accord              646
Tout à fait d’accord         327
Non réponse / Ne sait pas    282
Name: count, dtype: int64[pyarrow]


In [3]:
# Display the selected sociodemographic variables
codes = ["SEXE", "AGE_1", "DIPLOME", "ACTIVITE_REP", "REC_TailleAgglo5"]

for code in codes:
    column = next(col for col in hce.columns if col.startswith(code + " :"))
    print(column)

SEXE : SEXE - Vous êtes :
AGE_1 : AGE_1 (AGE - Veuillez nous indiquer votre âge :)
DIPLOME : DIPLOME - Quel est le diplôme le plus élevé que vous ayez obtenu ?
ACTIVITE_REP : ACTIVITE_REP - Actuellement, quelle est votre situation principale ?
REC_TailleAgglo5 : REC_TailleAgglo5


In [4]:
# Select and rename the variables
column_codes = {"SEXE": "sex","AGE_1": "age","DIPLOME": "education", "ACTIVITE_REP": "employment_status",
                "REC_TailleAgglo5": "urban_area_size","Q4_19": "target_response"}

selected_columns = {}

for code, name in column_codes.items():
    column = next(col for col in hce.columns if col.startswith(code + " :"))
    selected_columns[column] = name

ml_data = hce[list(selected_columns)].rename(columns=selected_columns).copy()

# Keep respondent IDs and survey weights
ml_data["respondent_id"] = hce["responseid"]
ml_data["survey_weight"] = pd.to_numeric(
    hce["POIDS"].str.replace(",", ".", regex=False),
    errors="raise")

# Create the binary target
target_mapping = {
    "Tout à fait d’accord": 1,
    "Plutôt d’accord": 1,
    "Plutôt pas d’accord": 0,
    "Pas du tout d’accord": 0}

ml_data["target"] = ml_data["target_response"].map(target_mapping)

print("Responses excluded:", ml_data["target"].isna().sum())

# Keep respondents who expressed agreement or disagreement
ml_data = ml_data.dropna(subset=["target"]).copy()
ml_data["target"] = ml_data["target"].astype(int)
ml_data["age"] = pd.to_numeric(ml_data["age"], errors="raise")

print("Usable respondents:", len(ml_data))
print("\nTarget counts:")
print(ml_data["target"].value_counts().sort_index())

Responses excluded: 282
Usable respondents: 2779

Target counts:
target
0    1806
1     973
Name: count, dtype: int64


In [5]:
# Check the data before modeling
features = ["sex", "age", "education", "employment_status", "urban_area_size"]

print("Missing predictor values:")
print(ml_data[features].isna().sum())

print("\nMissing respondent IDs:", ml_data["respondent_id"].isna().sum())
print("Duplicate respondent IDs:", ml_data["respondent_id"].duplicated().sum())

print("\nMissing survey weights:", ml_data["survey_weight"].isna().sum())
print("Zero or negative weights:", ml_data["survey_weight"].le(0).sum())

print("\nAge range:", ml_data["age"].min(), "to", ml_data["age"].max())

Missing predictor values:
sex                  0
age                  0
education            0
employment_status    0
urban_area_size      0
dtype: int64

Missing respondent IDs: 0
Duplicate respondent IDs: 0

Missing survey weights: 0
Zero or negative weights: 0

Age range: 15 to 90


### Data preparation results

The dataset contains 2,779 respondents who expressed an opinion:
- 1,806 disagreed with the statement.
- 973 agreed with the statement.

The 282 non-response or "Don't know" answers were excluded.
These counts are not survey-weighted.

The selected predictors have no missing values.
Respondent IDs are complete and unique, and survey weights are positive.
Respondents are aged from 15 to 90.

The respondent ID and survey weight will not be used as predictors.

## Step 2 — Train / Test Split

Split the data into a training set and a test set.

The training set will be used to build and tune the models.
The test set will be used to evaluate them.

In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# Separate predictors, target and survey weights
X = ml_data[features].copy()
y = ml_data["target"]
survey_weights = ml_data["survey_weight"].astype(float)

# Split them together to keep each respondent's data aligned
X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(
    X, y, survey_weights,
    test_size=0.20,
    random_state=SEED,
    stratify=y)

print("Training respondents:", len(X_train))
print("Test respondents:", len(X_test))
print("Shared respondents:", len(set(X_train.index) & set(X_test.index)))

Training respondents: 2223
Test respondents: 556
Shared respondents: 0


## Step 3 — Feature Engineering

Prepare the predictors for the models:

- Standardize age.
- Encode categorical variables using one-hot encoding.

The transformations will be learned from the training data.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prepare numerical and categorical variables
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), ["age"]),
    ("categorical", OneHotEncoder(
        drop="first", handle_unknown="ignore", sparse_output=False
    ), ["sex", "education", "employment_status", "urban_area_size"])
])

print("Preprocessor ready")

Preprocessor ready


## Baseline model

Before testing more advanced models, I will use a simple baseline.
It predicts the most frequent response and gives a reference point
for comparing the other models.


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score

# Predict the most frequent class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train, sample_weight=weights_train)

# Evaluate on train and test
baseline_train_pred = baseline.predict(X_train)
baseline_test_pred = baseline.predict(X_test)

print("Train balanced accuracy:",
      round(balanced_accuracy_score(y_train, baseline_train_pred,
                                    sample_weight=weights_train), 3))

print("Test balanced accuracy:",
      round(balanced_accuracy_score(y_test, baseline_test_pred,
                                    sample_weight=weights_test), 3))

Train balanced accuracy: 0.5
Test balanced accuracy: 0.5


## Logistic Regression — SGD Classifier

### Step 4 — Model Training

Train a logistic regression model to predict agreement or disagreement.
The model uses the survey weights during training.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import SGDClassifier

# Create the logistic regression model
logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", SGDClassifier(loss="log_loss",alpha=0.01,class_weight="balanced",max_iter=5000,random_state=SEED))])

# Train the model using survey weights
logistic_model.fit(X_train, y_train,model__sample_weight=weights_train)

print("Logistic regression trained")

Logistic regression trained


In [10]:
from sklearn.metrics import roc_auc_score

# Predict responses for train and test
logistic_train_pred = logistic_model.predict(X_train)
logistic_test_pred = logistic_model.predict(X_test)

# Compare balanced accuracy
train_score = balanced_accuracy_score(
    y_train, logistic_train_pred, sample_weight=weights_train)

test_score = balanced_accuracy_score(
    y_test, logistic_test_pred, sample_weight=weights_test)

# Check how well the model ranks the two classes
test_probabilities = logistic_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_probabilities, sample_weight=weights_test)

print("Train balanced accuracy:", round(train_score, 3))
print("Test balanced accuracy:", round(test_score, 3))
print("Test ROC AUC:", round(test_auc, 3))

Train balanced accuracy: 0.587
Test balanced accuracy: 0.579
Test ROC AUC: 0.57


In [11]:
from sklearn.metrics import confusion_matrix

# Show the number of respondents in each prediction group
logistic_cm = confusion_matrix(y_test, logistic_test_pred)

display(pd.DataFrame(
    logistic_cm,
    index=["Actual disagreement", "Actual agreement"],
    columns=["Predicted disagreement", "Predicted agreement"]))

,Predicted disagreement,Predicted agreement
Actual disagreement,192,169
Actual agreement,73,122


### Step 6 — Model Analysis

The model reached a survey-weighted balanced accuracy of 0.587
on the training set and 0.579 on the test set.

These scores are close, but performance remains modest compared
with the baseline score of 0.500.

The confusion matrix shows errors in both classes. The model is
not reliable enough to predict an individual respondent's opinion.

## Random Forest Classifier

### Step 4 — Model Training

Train a Random Forest classifier and evaluate its performance.

The model uses survey weights during training.

In [12]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest model
forest_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=1))])

# Train the model using survey weights
forest_model.fit(
    X_train, y_train,
    model__sample_weight=weights_train)

print("Random Forest trained")

Random Forest trained


In [13]:
# Predict responses for train and test
forest_train_pred = forest_model.predict(X_train)
forest_test_pred = forest_model.predict(X_test)

# Compare balanced accuracy
forest_train_score = balanced_accuracy_score(
    y_train, forest_train_pred, sample_weight=weights_train)

forest_test_score = balanced_accuracy_score(
    y_test, forest_test_pred, sample_weight=weights_test)

# Calculate ROC AUC on the test set
forest_test_prob = forest_model.predict_proba(X_test)[:, 1]
forest_test_auc = roc_auc_score(
    y_test, forest_test_prob, sample_weight=weights_test)

print("Train balanced accuracy:", round(forest_train_score, 3))
print("Test balanced accuracy:", round(forest_test_score, 3))
print("Test ROC AUC:", round(forest_test_auc, 3))

Train balanced accuracy: 0.5
Test balanced accuracy: 0.5
Test ROC AUC: 0.574


In [14]:
# Compare actual and predicted classes on the test set
print("Actual class counts:")
print(y_test.value_counts().sort_index())

print("\nPredicted class counts:")
print(pd.Series(forest_test_pred).value_counts().sort_index())

Actual class counts:
target
0    361
1    195
Name: count, dtype: int64

Predicted class counts:
0    556
Name: count, dtype: int64


### Step 6 — Model Analysis

The Random Forest predicts disagreement for every respondent
in the test set. Its balanced accuracy is therefore 0.500,
the same as the baseline.

The test ROC AUC is 0.574. This suggests a limited ability
to rank responses using predicted probabilities, even though
the current decision threshold predicts only one class.

Further tuning will be explored using the training data.

In [15]:
from sklearn.model_selection import StratifiedKFold, ParameterGrid

# Create the same three folds as in the draft notebook
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Test a small grid of settings
grid = {
    "model__max_depth": [5, 10],
    "model__min_samples_leaf": [1, 5],
    "model__class_weight": [None, "balanced"]
}

forest_results = []

for params in ParameterGrid(grid):
    train_scores = []
    valid_scores = []

    for train_index, valid_index in cv.split(X_train, y_train):
        candidate = clone(forest_model)
        candidate.set_params(**params)

        candidate.fit(
            X_train.iloc[train_index],
            y_train.iloc[train_index],
            model__sample_weight=weights_train.iloc[train_index])

        train_pred = candidate.predict(X_train.iloc[train_index])
        valid_pred = candidate.predict(X_train.iloc[valid_index])

        train_scores.append(balanced_accuracy_score(
            y_train.iloc[train_index], train_pred,
            sample_weight=weights_train.iloc[train_index]))
        valid_scores.append(balanced_accuracy_score(
            y_train.iloc[valid_index], valid_pred,
            sample_weight=weights_train.iloc[valid_index]))

    forest_results.append({
        "max_depth": params["model__max_depth"],
        "min_samples_leaf": params["model__min_samples_leaf"],
        "class_weight": str(params["model__class_weight"]),
        "train_balanced_accuracy": sum(train_scores) / len(train_scores),
        "valid_balanced_accuracy": sum(valid_scores) / len(valid_scores)})

forest_tuning = pd.DataFrame(forest_results)
display(forest_tuning.sort_values("valid_balanced_accuracy",
                                  ascending=False).round(3))

,max_depth,min_samples_leaf,class_weight,train_balanced_accuracy,valid_balanced_accuracy
4,5,1,balanced,0.640,0.571
5,5,5,balanced,0.620,0.569
7,10,5,balanced,0.676,0.555
6,10,1,balanced,0.815,0.542
2,10,1,None,0.692,0.509
3,10,5,None,0.547,0.506
1,5,5,None,0.502,0.502
0,5,1,None,0.510,0.500


In [16]:
# Train the best Random Forest configuration on the full training set
forest_tuned = clone(forest_model)

forest_tuned.set_params(
    model__max_depth=5,
    model__min_samples_leaf=1,
    model__class_weight="balanced")

forest_tuned.fit(
    X_train, y_train,
    model__sample_weight=weights_train)

# Evaluate on the test set
forest_tuned_pred = forest_tuned.predict(X_test)
forest_tuned_prob = forest_tuned.predict_proba(X_test)[:, 1]

print("Tuned Random Forest test balanced accuracy:",
      round(balanced_accuracy_score(
          y_test, forest_tuned_pred, sample_weight=weights_test), 3))

print("Tuned Random Forest test ROC AUC:",
      round(roc_auc_score(
          y_test, forest_tuned_prob, sample_weight=weights_test), 3))

Tuned Random Forest test balanced accuracy: 0.585
Tuned Random Forest test ROC AUC: 0.575


In [17]:
# Show the number of respondents in each prediction group
forest_tuned_cm = confusion_matrix(y_test, forest_tuned_pred)

display(pd.DataFrame(
    forest_tuned_cm,
    index=["Actual disagreement", "Actual agreement"],
    columns=["Predicted disagreement", "Predicted agreement"]))

,Predicted disagreement,Predicted agreement
Actual disagreement,213,148
Actual agreement,82,113


### Tuned Random Forest — Model Analysis

I tested eight combinations of hyperparameters using three-fold
cross-validation on the training set.

The best configuration used:
- max_depth = 5
- min_samples_leaf = 1
- class_weight = "balanced"

Its mean training balanced accuracy was 0.640, compared with 0.571
in validation. This gap suggests some overfitting.

After fitting this configuration on the full training set, the model
reached a test balanced accuracy of 0.585 and a test ROC AUC of 0.575.

It now predicts both classes. However, errors remain frequent:
148 respondents who disagreed were predicted to agree, and
82 respondents who agreed were predicted to disagree.

Its test scores are slightly higher than those of logistic regression,
but this small difference does not establish a clear advantage.

## XGBoost Classifier

### Step 4 — Model Training

Train an XGBoost classifier using survey weights.

Unlike Random Forest, XGBoost builds trees sequentially.
Each new tree aims to improve the model's predictions.

In [18]:
from xgboost import XGBClassifier

# Create the XGBoost model
xgb_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
        n_jobs=1,
        eval_metric="logloss"
    ))
])

# Train the model using survey weights
xgb_model.fit(
    X_train, y_train,
    model__sample_weight=weights_train
)

print("XGBoost trained")

XGBoost trained


In [19]:
# Predict responses for train and test
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Compare balanced accuracy
xgb_train_score = balanced_accuracy_score(
    y_train, xgb_train_pred, sample_weight=weights_train)

xgb_test_score = balanced_accuracy_score(
    y_test, xgb_test_pred, sample_weight=weights_test)

# Calculate ROC AUC on the test set
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_test_auc = roc_auc_score(
    y_test, xgb_test_prob, sample_weight=weights_test)

print("Train balanced accuracy:", round(xgb_train_score, 3))
print("Test balanced accuracy:", round(xgb_test_score, 3))
print("Test ROC AUC:", round(xgb_test_auc, 3))

Train balanced accuracy: 0.569
Test balanced accuracy: 0.522
Test ROC AUC: 0.578


In [20]:
# Inspect XGBoost predictions on the test set
print("Predicted class counts:")
print(pd.Series(xgb_test_pred).value_counts().sort_index())

display(pd.DataFrame(confusion_matrix(y_test, xgb_test_pred),
    index=["Actual disagreement", "Actual agreement"],
    columns=["Predicted disagreement", "Predicted agreement"]))

Predicted class counts:
0    506
1     50
Name: count, dtype: int64


,Predicted disagreement,Predicted agreement
Actual disagreement,334,27
Actual agreement,172,23


In [21]:
# Compare class balancing and tree depth using cross-validation
xgb_results = []

for depth in [2, 3]:
    for balance in [False, True]:
        train_scores = []
        valid_scores = []

        for train_index, valid_index in cv.split(X_train, y_train):
            y_fold = y_train.iloc[train_index]
            weights_fold = weights_train.iloc[train_index]

            class_ratio = 1

            if balance:
                weight_0 = weights_fold[y_fold == 0].sum()
                weight_1 = weights_fold[y_fold == 1].sum()
                class_ratio = weight_0 / weight_1

            candidate = clone(xgb_model)
            candidate.set_params(
                model__max_depth=depth,
                model__scale_pos_weight=class_ratio)

            candidate.fit(
                X_train.iloc[train_index], y_fold,
                model__sample_weight=weights_fold)

            train_pred = candidate.predict(X_train.iloc[train_index])
            valid_pred = candidate.predict(X_train.iloc[valid_index])

            train_scores.append(balanced_accuracy_score(
                y_fold, train_pred, sample_weight=weights_fold))
            valid_scores.append(balanced_accuracy_score(
                y_train.iloc[valid_index], valid_pred,
                sample_weight=weights_train.iloc[valid_index]))

        xgb_results.append({
            "max_depth": depth,
            "class_balancing": balance,
            "train_balanced_accuracy": sum(train_scores) / len(train_scores),
            "valid_balanced_accuracy": sum(valid_scores) / len(valid_scores)})

xgb_tuning = pd.DataFrame(xgb_results)
display(xgb_tuning.sort_values("valid_balanced_accuracy",ascending=False).round(3))

,max_depth,class_balancing,train_balanced_accuracy,valid_balanced_accuracy
1,2,True,0.635,0.566
3,3,True,0.675,0.560
2,3,False,0.584,0.515
0,2,False,0.546,0.508


In [22]:
# Calculate class balancing from the full training set
weight_0 = weights_train[y_train == 0].sum()
weight_1 = weights_train[y_train == 1].sum()
class_ratio = weight_0 / weight_1

# Train the selected XGBoost configuration
xgb_tuned = clone(xgb_model)
xgb_tuned.set_params(model__max_depth=2,
    model__scale_pos_weight=class_ratio)

xgb_tuned.fit(X_train, y_train,model__sample_weight=weights_train)

# Evaluate on the test set
xgb_tuned_pred = xgb_tuned.predict(X_test)
xgb_tuned_prob = xgb_tuned.predict_proba(X_test)[:, 1]

print("Tuned XGBoost test balanced accuracy:",
      round(balanced_accuracy_score(y_test, xgb_tuned_pred, sample_weight=weights_test), 3))

print("Tuned XGBoost test ROC AUC:",round(roc_auc_score(y_test, xgb_tuned_prob, sample_weight=weights_test), 3))

Tuned XGBoost test balanced accuracy: 0.571
Tuned XGBoost test ROC AUC: 0.569


In [23]:
# Inspect errors from the tuned XGBoost model
display(pd.DataFrame(
    confusion_matrix(y_test, xgb_tuned_pred),
    index=["Actual disagreement", "Actual agreement"],
    columns=["Predicted disagreement", "Predicted agreement"]))

,Predicted disagreement,Predicted agreement
Actual disagreement,201,160
Actual agreement,81,114


In [24]:
from sklearn.model_selection import ParameterSampler

# Randomly select five combinations of XGBoost settings
parameters = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [1, 2, 3],
    "reg_lambda": [1, 5, 10]}

random_settings = list(ParameterSampler(
    parameters, n_iter=5, random_state=SEED))

xgb_random_results = []

for settings in random_settings:
    train_scores = []
    valid_scores = []

    for train_index, valid_index in cv.split(X_train, y_train):
        y_fold = y_train.iloc[train_index]
        weights_fold = weights_train.iloc[train_index]

        weight_0 = weights_fold[y_fold == 0].sum()
        weight_1 = weights_fold[y_fold == 1].sum()

        candidate = clone(xgb_model)
        candidate.set_params(**{"model__" + name: value for name, value in settings.items()})
        candidate.set_params(model__scale_pos_weight=weight_0 / weight_1)

        candidate.fit(X_train.iloc[train_index], y_fold,
            model__sample_weight=weights_fold)

        train_pred = candidate.predict(X_train.iloc[train_index])
        valid_pred = candidate.predict(X_train.iloc[valid_index])

        train_scores.append(balanced_accuracy_score(
            y_fold, train_pred, sample_weight=weights_fold))
        valid_scores.append(balanced_accuracy_score(
            y_train.iloc[valid_index], valid_pred,
            sample_weight=weights_train.iloc[valid_index]))

    result = settings.copy()
    result["train_balanced_accuracy"] = sum(train_scores) / len(train_scores)
    result["valid_balanced_accuracy"] = sum(valid_scores) / len(valid_scores)
    xgb_random_results.append(result)

xgb_random_search = pd.DataFrame(xgb_random_results)
xgb_random_search = xgb_random_search.sort_values("valid_balanced_accuracy", ascending=False)

display(xgb_random_search.round(3))

,reg_lambda,n_estimators,max_depth,learning_rate,train_balanced_accuracy,valid_balanced_accuracy
1,1,100,1,0.03,0.587,0.581
3,5,200,1,0.05,0.591,0.579
0,1,200,1,0.05,0.594,0.577
4,1,100,3,0.03,0.624,0.565
2,5,200,3,0.03,0.646,0.564


In [25]:
# Train the best configuration from the random search
xgb_random_best = clone(xgb_model)
xgb_random_best.set_params(
    model__n_estimators=100,
    model__max_depth=1,
    model__learning_rate=0.03,
    model__reg_lambda=1,
    model__scale_pos_weight=(
        weights_train[y_train == 0].sum()
        / weights_train[y_train == 1].sum()))

xgb_random_best.fit(
    X_train, y_train,
    model__sample_weight=weights_train)

# Evaluate on the test set
xgb_random_test_pred = xgb_random_best.predict(X_test)
xgb_random_test_prob = xgb_random_best.predict_proba(X_test)[:, 1]

print("Random search XGBoost test balanced accuracy:",
      round(balanced_accuracy_score(
          y_test, xgb_random_test_pred, sample_weight=weights_test), 3))

print("Random search XGBoost test ROC AUC:",
      round(roc_auc_score(
          y_test, xgb_random_test_prob, sample_weight=weights_test), 3))

Random search XGBoost test balanced accuracy: 0.579
Random search XGBoost test ROC AUC: 0.582


### XGBoost — Model Analysis

The initial XGBoost model reached a test balanced accuracy of 0.522.
It predicted disagreement for 506 of the 556 test respondents.

I first compared four configurations using three-fold cross-validation
on the training set, adjusting tree depth and class balancing.
The best validation balanced accuracy was 0.566.

I then used random search to evaluate five additional configurations.
The best combination used:
- n_estimators = 100
- max_depth = 1
- learning_rate = 0.03
- reg_lambda = 1
- Class balancing through scale_pos_weight

This configuration reached a mean training balanced accuracy of 0.587
and a mean validation balanced accuracy of 0.581.
The small gap suggests limited overfitting in these folds.

After fitting it on the full training set, test balanced accuracy
was 0.579 and test ROC AUC was 0.582.

Performance remains modest. The same test set was used to evaluate
several models, so comparisons are exploratory rather than a fully
independent final assessment. Reorganizing the draft into this notebook
does not create a new, unseen test set.

In [26]:
# Compare the models on the same test set
model_comparison = pd.DataFrame({"model": ["Baseline","Logistic regression",
        "Tuned Random Forest", "Random search XGBoost"],
    "test_balanced_accuracy": [
        balanced_accuracy_score( y_test, baseline_test_pred, sample_weight=weights_test),
        balanced_accuracy_score( y_test, logistic_test_pred, sample_weight=weights_test),
        balanced_accuracy_score(y_test, forest_tuned_pred, sample_weight=weights_test),
        balanced_accuracy_score( y_test, xgb_random_test_pred, sample_weight=weights_test )],
    "test_roc_auc": [ 0.5,
        roc_auc_score(
            y_test, logistic_model.predict_proba(X_test)[:, 1],
            sample_weight=weights_test),
        roc_auc_score(
            y_test, forest_tuned_prob, sample_weight=weights_test),
        roc_auc_score(
            y_test, xgb_random_test_prob, sample_weight=weights_test)]})

display(model_comparison.round(3))

,model,test_balanced_accuracy,test_roc_auc
0,Baseline,0.500,0.500
1,Logistic regression,0.579,0.570
2,Tuned Random Forest,0.585,0.575
3,Random search XGBoost,0.579,0.582


## Model Comparison

All three models performed above the baseline on this test set,
but their predictive performance remained modest.

Tuned Random Forest achieved the highest test balanced accuracy (0.585).
XGBoost after random search achieved the highest test ROC AUC (0.582).

XGBoost also achieved a higher cross-validation balanced accuracy
than the tuned Random Forest: 0.581 compared with 0.571.

No model clearly dominated across all evaluation results.
Random Forest was retained for the demonstration because it achieved
the highest test balanced accuracy, the project's main metric.

The differences are small, and the same test set was used repeatedly.
The comparison should therefore be interpreted cautiously.

In [27]:
# Compare two groups of predictors with the same logistic model
feature_groups = {
    "Sex and age": ["sex", "age"],
    "All five features": features
}

feature_results = []

for group_name, columns in feature_groups.items():
    categorical_columns = [col for col in columns if col != "age"]

    group_preprocessor = ColumnTransformer([
        ("numeric", StandardScaler(), ["age"]),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_columns)])

    group_model = Pipeline([
        ("preprocessor", group_preprocessor),
        ("model", clone(logistic_model.named_steps["model"]))])

    valid_scores = []

    for train_index, valid_index in cv.split(X_train, y_train):
        candidate = clone(group_model)

        candidate.fit(
            X_train[columns].iloc[train_index],
            y_train.iloc[train_index],
            model__sample_weight=weights_train.iloc[train_index] )

        predictions = candidate.predict(X_train[columns].iloc[valid_index])

        valid_scores.append(balanced_accuracy_score(
            y_train.iloc[valid_index], predictions,
            sample_weight=weights_train.iloc[valid_index] ))

    feature_results.append({ "features": group_name,
        "valid_balanced_accuracy": sum(valid_scores) / len(valid_scores) })

feature_comparison = pd.DataFrame(feature_results)
display(feature_comparison.round(3))

,features,valid_balanced_accuracy
0,Sex and age,0.585
1,All five features,0.577


## Feature Selection Experiment

I compared two groups of predictors using the same logistic regression
settings, survey weights and three validation folds.

Sex and age reached a mean validation balanced accuracy of 0.585,
compared with 0.577 for all five predictors.

Adding education, employment status and urban area size did not improve
the score in this experiment. However, the difference is small and does
not establish that these variables have no relationship with the response.

This comparison concerns one model with fixed settings.

## Model Retained for the Demonstration

I retained the tuned Random Forest for the educational demonstration.
It achieved the highest test balanced accuracy (0.585), which is
the main evaluation metric used in this project.

However, its advantage was small. XGBoost achieved a higher ROC AUC
and a higher cross-validation balanced accuracy.

There is therefore no clear overall winner. The repeated use of the
same test set also limits the strength of this comparison.

The demonstration illustrates how the model works. Its modest
performance does not support individual profiling or operational use.

In [28]:
from pathlib import Path
import joblib

# Save the selected Random Forest and its preprocessing
model_folder = Path("models")
model_folder.mkdir(exist_ok=True)

model_path = model_folder / "hce_agreement_classifier.joblib"
joblib.dump(forest_tuned, model_path)

print("Saved model: Tuned Random Forest")
print("Saved:", model_path)

Saved model: Tuned Random Forest
Saved: models\hce_agreement_classifier.joblib


In [29]:
# Save model comparison results for Streamlit
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

comparison_path = output_folder / "hce_model_comparison.csv"
model_comparison.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)

Saved: data\processed\hce_model_comparison.csv
